# Week 5 Semantic Search Exploration

This notebook validates the Week 5 deliverables on a sample of collected listing data.

## Deliverables tracked
- Sentence embeddings for listing remarks (384 or 768 dims)
- FAISS index for efficient similarity search
- `SemanticSearcher` with query embedding + retrieval
- Semantic vs BM25 comparison study
- Latency `< 100ms` for 10k listings
- Relevance evaluation set with 50 query-result pairs

In [12]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.semantic_searcher import (
    BM25Searcher,
    SemanticSearcher,
    benchmark_latency,
    build_comparison_table,
    build_relevance_pairs,
    summarize_relevance_scores,
)

DATASET_PATH = PROJECT_ROOT / "data/processed/listing_sample_cleaned.csv"
QUERIES_PATH = PROJECT_ROOT / "data/processed/user_queries.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET_PATH.exists())
print("Queries exists:", QUERIES_PATH.exists())

Project root: /Users/nathanye/Documents/GitHub/nlp-internship
Dataset exists: True
Queries exists: True


In [13]:
df = pd.read_csv(DATASET_PATH)
queries_df = pd.read_csv(QUERIES_PATH)

remarks = df["remarks"].fillna("").astype(str).tolist()
listing_ids = df["L_ListingID"].astype(str).tolist() if "L_ListingID" in df.columns else [str(i) for i in range(len(df))]
queries = queries_df["query"].dropna().astype(str).tolist()

print(f"Listings loaded: {len(remarks)}")
print(f"Queries loaded: {len(queries)}")
df[["remarks"]].head(3)

Listings loaded: 1000
Queries loaded: 58


,remarks
0,Successful vacation rental just minutes from S...
1,"This must-see home, less than five years old, ..."
2,New construction with an ADU in the heart of U...


In [14]:
# 1) Sentence embeddings + 2) FAISS index + 3) SemanticSearcher retrieval
semantic = SemanticSearcher(model_name="sentence-transformers/all-MiniLM-L6-v2")
semantic.build_index(remarks, listing_ids=listing_ids)

print("Embedding dim:", semantic.embedding_dim)
print("Target dim met:", semantic.embedding_dim in {384, 768})
print("FAISS index type:", type(semantic.index).__name__)
print("Semantic searcher ready:", semantic.is_ready)

sample_query = "modern condo with garage near downtown"
semantic_results = semantic.search(sample_query, top_k=5)
pd.DataFrame([
    {"rank": i + 1, "listing_id": r.listing_id, "score": r.score, "remark": r.remark[:160]}
    for i, r in enumerate(semantic_results)
])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384
Target dim met: True
FAISS index type: IndexFlatIP
Semantic searcher ready: True


,rank,listing_id,score,remark
0,1,1150762115,0.589592,"Enjoy life in this rarely available, top floor..."
1,2,1126591879,0.589209,Price Reduced!!! This stylish two-story condo ...
2,3,1147559067,0.569756,"This updated 2-bedroom, 1-bath condo is tucked..."
3,4,1147175810,0.567098,Modern newer-construction multi-level townhome...
4,5,1107116525,0.555610,Welcome to the Old Orchard Condos! This second...


In [15]:
# 4) Comparison study: semantic vs BM25 keyword search
bm25 = BM25Searcher()
bm25.build_index(remarks, listing_ids=listing_ids)

study_queries = queries[:10]
comparison_df = build_comparison_table(semantic, bm25, study_queries, top_k=5)

print("Comparison rows:", len(comparison_df))
print("Methods included:", sorted(comparison_df["method"].unique().tolist()))
comparison_df.head(12)

Comparison rows: 100
Methods included: ['bm25', 'semantic']


,query,method,rank,listing_id,score,remark
0,3 bedroom home with pool in Los Angeles,semantic,1,1136722171,0.699300,Located in a highly desirable North Hollywood ...
1,3 bedroom home with pool in Los Angeles,semantic,2,1139320214,0.680614,Charming Spanish-style Pool home situated on ...
2,3 bedroom home with pool in Los Angeles,semantic,3,1003628672,0.668380,This very spacious 4 bedroom La Jolla Woods ho...
3,3 bedroom home with pool in Los Angeles,semantic,4,1146434487,0.642659,Huge Price Reduction! Must Sell Today! Beautif...
4,3 bedroom home with pool in Los Angeles,semantic,5,1147068590,0.635141,Looking for a big home with a big lot a Swimmi...
5,3 bedroom home with pool in Los Angeles,bm25,1,1134424128,12.921986,Seize this exceptional opportunity with 2 SFRs...
6,3 bedroom home with pool in Los Angeles,bm25,2,1150727123,12.239290,Welcome to Unit 421 at Vero Lofts a spacious ...
7,3 bedroom home with pool in Los Angeles,bm25,3,1150559667,12.142263,"Welcome to this classic 1908 Los Angeles home,..."
8,3 bedroom home with pool in Los Angeles,bm25,4,1149851144,11.436811,Welcome to this stunning high ceiling loft on ...
9,3 bedroom home with pool in Los Angeles,bm25,5,1145630409,10.641418,This is a beautifully maintained 3 bedrooms / ...


In [16]:
# 5) Latency benchmark: target < 100ms for 10k listings

def repeat_to_size(items, target_size):
    if not items:
        return []
    if len(items) >= target_size:
        return items[:target_size]
    out = list(items)
    i = 0
    while len(out) < target_size:
        out.append(items[i % len(items)])
        i += 1
    return out

remarks_10k = repeat_to_size(remarks, 10_000)
ids_10k = [str(i) for i in range(len(remarks_10k))]
queries_100 = repeat_to_size(queries, 100)

semantic_10k = SemanticSearcher(
    model_name=semantic.model_name,
    encoder=semantic._ensure_encoder(),
    batch_size=semantic.batch_size,
)
semantic_10k.build_index(remarks_10k, listing_ids=ids_10k)
latency_stats = benchmark_latency(semantic_10k, queries_100, top_k=10, warmup=10)

latency_table = pd.DataFrame([
    {"metric": "mean_ms", "value": latency_stats.mean_ms},
    {"metric": "p95_ms", "value": latency_stats.p95_ms},
    {"metric": "p99_ms", "value": latency_stats.p99_ms},
    {"metric": "max_ms", "value": latency_stats.max_ms},
])

latency_target_met = latency_stats.p95_ms < 100.0
print("Latency target met (<100ms p95):", latency_target_met)
latency_table

Latency target met (<100ms p95): True


,metric,value
0,mean_ms,12.720265
1,p95_ms,15.105654
2,p99_ms,23.232741
3,max_ms,24.028000


In [17]:
# 6) Relevance evaluation set: 50 query-result pairs
relevance_pairs_df = build_relevance_pairs(
    comparison_df,
    per_method_per_query=3,  # semantic + bm25 per query
    target_pairs=50,
)

print("Relevance pairs created:", len(relevance_pairs_df))
print("Required 50 met:", len(relevance_pairs_df) == 50)
relevance_pairs_df.head(10)

Relevance pairs created: 50
Required 50 met: True


,query,method,rank,listing_id,score,remark,relevant
0,3 bedroom home with pool in Los Angeles,semantic,1,1136722171,0.699300,Located in a highly desirable North Hollywood ...,
1,3 bedroom home with pool in Los Angeles,semantic,2,1139320214,0.680614,Charming Spanish-style Pool home situated on ...,
2,3 bedroom home with pool in Los Angeles,semantic,3,1003628672,0.668380,This very spacious 4 bedroom La Jolla Woods ho...,
3,3 bedroom home with pool in Los Angeles,bm25,1,1134424128,12.921986,Seize this exceptional opportunity with 2 SFRs...,
4,3 bedroom home with pool in Los Angeles,bm25,2,1150727123,12.239290,Welcome to Unit 421 at Vero Lofts a spacious ...,
5,3 bedroom home with pool in Los Angeles,bm25,3,1150559667,12.142263,"Welcome to this classic 1908 Los Angeles home,...",
6,Modern condo near downtown with gym,semantic,1,1114216202,0.569339,Very spacious and bright condo with a contempo...,
7,Modern condo near downtown with gym,semantic,2,1151678074,0.565120,This move in ready 1-bedroom corner unit is lo...,
8,Modern condo near downtown with gym,semantic,3,1149484197,0.547835,Step into a freshly reimagined haven nestled r...,
9,Modern condo near downtown with gym,bm25,1,1133715870,9.992916,Large 2 unit condo near the heart of Downtown ...,


## Relevance evaluation summary (from labeled CSV)

This section loads the labeled file and computes method-level relevance metrics.

Expected input file:
- `data/processed/relevance_pairs_export.csv`

Expected columns:
- `query`, `method`, `relevant` (0/1)

If the file is auto-labeled, these are heuristic labels; for final reporting, review and adjust manually as needed.

In [21]:
labeled_path = PROJECT_ROOT / "data/processed/relevance_pairs_export.csv"
labeled_df = pd.read_csv(labeled_path)

print("Loaded labeled file:", labeled_path)
print("Rows:", len(labeled_df))
print("Methods:", sorted(labeled_df["method"].dropna().unique().tolist()))
print("Relevant value counts:\n", labeled_df["relevant"].value_counts(dropna=False))

relevance_summary_df = summarize_relevance_scores(labeled_df)
relevance_summary_df

Loaded labeled file: /Users/nathanye/Documents/GitHub/nlp-internship/data/processed/relevance_pairs_export.csv
Rows: 50
Methods: ['bm25', 'semantic']
Relevant value counts:
 relevant
1    36
0    14
Name: count, dtype: int64


,method,num_pairs,mean_relevance,relevant_rate
0,bm25,24,0.666667,1.0
1,semantic,26,0.769231,1.0


## Acceptance Demo: Does semantic search *actually* work?

This section uses **real samples from the dataset** and compares top semantic results vs BM25 for paraphrase/synonym-heavy queries.

Interpretation guide:
- `semantic_top1_has_theme`: whether semantic top-1 contains at least one expected concept token.
- `bm25_top1_has_theme`: same check for BM25.
- Then inspect the real top-3 snippets from each method.

In [19]:
# Build searchers if this section is run independently
if "semantic" not in globals():
    semantic = SemanticSearcher(model_name="sentence-transformers/all-MiniLM-L6-v2")
    semantic.build_index(remarks, listing_ids=listing_ids)

if "bm25" not in globals():
    bm25 = BM25Searcher()
    bm25.build_index(remarks, listing_ids=listing_ids)

acceptance_queries = [
    {"query": "home with swimming area", "theme_tokens": ["pool", "swim"]},
    {"query": "ocean-facing condo", "theme_tokens": ["ocean", "waterfront", "view"]},
    {"query": "car space or covered parking", "theme_tokens": ["garage", "carport", "parking"]},
    {"query": "newly renovated move-in-ready home", "theme_tokens": ["remodel", "updated", "move-in", "renovat"]},
    {"query": "quiet street close to parks", "theme_tokens": ["quiet", "park", "cul-de-sac"]},
    {"query": "home office setup", "theme_tokens": ["office", "den", "workspace"]},
    {"query": "large backyard for pets", "theme_tokens": ["yard", "backyard", "fenced", "pet"]},
    {"query": "energy efficient house", "theme_tokens": ["solar", "energy", "efficient"]},
    {"query": "walking distance to schools", "theme_tokens": ["school", "district", "elementary"]},
    {"query": "downtown lifestyle condo", "theme_tokens": ["downtown", "urban", "city"]},
]

rows = []
detail_frames = []

for spec in acceptance_queries:
    q = spec["query"]
    theme_tokens = [t.lower() for t in spec["theme_tokens"]]

    sem = semantic.search(q, top_k=3)
    bm = bm25.search(q, top_k=3)

    sem_top = sem[0].remark if sem else ""
    bm_top = bm[0].remark if bm else ""

    sem_hit = any(tok in sem_top.lower() for tok in theme_tokens)
    bm_hit = any(tok in bm_top.lower() for tok in theme_tokens)

    rows.append({
        "query": q,
        "expected_theme_tokens": ", ".join(theme_tokens),
        "semantic_top1_has_theme": sem_hit,
        "bm25_top1_has_theme": bm_hit,
        "semantic_top1_score": sem[0].score if sem else np.nan,
        "bm25_top1_score": bm[0].score if bm else np.nan,
        "semantic_top1_snippet": sem_top[:180],
        "bm25_top1_snippet": bm_top[:180],
    })

    for i, r in enumerate(sem, start=1):
        detail_frames.append({
            "query": q,
            "method": "semantic",
            "rank": i,
            "score": r.score,
            "listing_id": r.listing_id,
            "snippet": r.remark[:180],
        })
    for i, r in enumerate(bm, start=1):
        detail_frames.append({
            "query": q,
            "method": "bm25",
            "rank": i,
            "score": r.score,
            "listing_id": r.listing_id,
            "snippet": r.remark[:180],
        })

acceptance_summary_df = pd.DataFrame(rows)
acceptance_details_df = pd.DataFrame(detail_frames)

print("Semantic top-1 theme hit rate:", round(acceptance_summary_df["semantic_top1_has_theme"].mean(), 3))
print("BM25 top-1 theme hit rate:", round(acceptance_summary_df["bm25_top1_has_theme"].mean(), 3))
acceptance_summary_df

Semantic top-1 theme hit rate: 0.7
BM25 top-1 theme hit rate: 1.0


,query,expected_theme_tokens,semantic_top1_has_theme,bm25_top1_has_theme,semantic_top1_score,bm25_top1_score,semantic_top1_snippet,bm25_top1_snippet
0,home with swimming area,"pool, swim",True,True,0.596287,5.478583,Charming Spanish-style Pool home situated on ...,"Welcome to 8634 Hunt Canyon Road, a beautifull..."
1,ocean-facing condo,"ocean, waterfront, view",True,True,0.703090,10.631026,Conveniently located just minutes from Downtow...,NEW PRICE! ~ BEACH FRONT CONDO with PANORAMIC ...
2,car space or covered parking,"garage, carport, parking",False,True,0.447726,6.433652,"Discover this stunning 3-bedroom, 3-bathroom h...",Inviting entry leads to nicely upgraded 1BR/1B...
3,newly renovated move-in-ready home,"remodel, updated, move-in, renovat",False,True,0.585974,10.626980,Single-Story Living | Fully Furnished | No HOA...,SELLER FINANCING AVAILABLE! Discover this newl...
4,quiet street close to parks,"quiet, park, cul-de-sac",True,True,0.482339,10.286410,"Enjoy life in this rarely available, top floor...","Located in a desirable San Mateo neighborhood,..."
5,home office setup,"office, den, workspace",True,True,0.450938,6.954343,Single-family home with breathtaking golf cour...,"This charming three-bedroom, one-bath home off..."
6,large backyard for pets,"yard, backyard, fenced, pet",True,True,0.559450,6.255292,Beautifully maintained single-story home featu...,"Stunning Cul-de-Sac Gem! 3 Beds, 2 Baths, Huge..."
7,energy efficient house,"solar, energy, efficient",True,True,0.662354,9.290311,Beautiful 2-story home in our Rancho Madrina c...,"Brand new, energy-efficient home available by ..."
8,walking distance to schools,"school, district, elementary",True,True,0.356248,9.632132,Welcome to this elegantly remodeled single-sto...,SFR + ADU Opportunity in Prime Ontario Locatio...
9,downtown lifestyle condo,"downtown, urban, city",False,True,0.644391,8.029127,"Enjoy life in this rarely available, top floor...",Conveniently located just minutes from Downtow...


In [20]:
# Inspect real top-3 results for a specific query from the acceptance set
query_to_inspect = acceptance_queries[0]["query"]
acceptance_details_df[acceptance_details_df["query"] == query_to_inspect].sort_values(["method", "rank"])

,query,method,rank,score,listing_id,snippet
3,home with swimming area,bm25,1,5.478583,1144525929,"Welcome to 8634 Hunt Canyon Road, a beautifull..."
4,home with swimming area,bm25,2,5.124619,1150118930,"A smart alternative to traditional housing, th..."
5,home with swimming area,bm25,3,5.035793,1118294602,Welcome to your dream home in Woodbury East! T...
0,home with swimming area,semantic,1,0.596287,1139320214,Charming Spanish-style Pool home situated on ...
1,home with swimming area,semantic,2,0.574717,1151430999,Cute two bedroom one bath home with fully encl...
2,home with swimming area,semantic,3,0.574445,1147476041,"Stunning 4 Bedroom, 3 Bath Home in Highly Desi..."
